# 環境檢查: 上課一開始，大家一起把這一本跑完

這一本不教任何東西，它只做一件事: **逐項確認 `2_lab06_forecasting.ipynb` 與 `3_lab07_root_cause_analysis.ipynb` 需要的東西在這台機器上都有，而且真的跑得動。**

上課一開始會請所有人先跑這一本,當場把沒裝好的東西補起來。**前一天先跑過的人，當天只是再確認一次。**

## 怎麼用

1. 確認左邊的檔案清單裡看得到 `2_lab06_forecasting.ipynb` 與 `data` 這個資料夾。看不到就是資料夾開錯了,先關掉 JupyterLab,改在 `week6` 這個資料夾裡重開。
2. 上方選單按 **Run > Run All Cells**,然後等它跑完。
3. **每一格跑完就會印出自己那一組的結果**,不用等到最後。看到「失敗」就可以馬上開始修,不必等整本跑完。
4. 最後一格會把所有失敗的項目集中列出來,**照「修起來要多久」排序**,並附一段可以複製給講師的純文字。

狀態只有三種:

- **通過**: 這一項沒問題
- **失敗**: 這一項會讓 lab 跑不動,同一列的「沒過的話怎麼辦」寫了要打哪一行指令
- **選配**: 缺了不會擋住前面任何一格分析。目前只有 Docker 那一組是這個狀態

## 檢查的順序是照「修起來要多久」排的

這一點是刻意的。前面兩組如果沒過,修起來要下載東西,**越早開始越好**:

| 順序 | 檢查什麼 | 沒過的話要花多久修 |
| :--- | :--- | :--- |
| 1 | 機器與 Python、資料夾對不對 | 幾秒。開錯資料夾就關掉重開 |
| 2 | **Docker** | **最久。要下載並安裝 Docker Desktop,視網路 10 到 30 分鐘** |
| 3 | **套件** | **久。`conda env update` 要下載,5 到 20 分鐘** |
| 4 | Prophet 與 scikit-learn 真的配一次模型 | 中等。多半是重裝 prophet |
| 5 | 之後幾組 (字型、資料檔、寫入權限、教材檔案、JupyterLab、編碼) | 幾分鐘以內 |

所以看到第 2 組或第 3 組失敗,**先把那一行指令貼到終端機開始跑**,再回來看後面的結果,不要乾等。

## 兩件事先講清楚

- **這一本不會中斷。** 每一項檢查都包在 try/except 裡,任何一項出錯都只會變成表格裡的一列,後面的檢查照跑。所以看到紅底的錯誤訊息才是意外,正常情況下應該一路跑到最後一格。
- **Prophet 那一格會實際配適一次模型。** 它背後的 cmdstanpy 要先把統計模型載入才能算,所以這一格比別格久,它會印出這台機器上實際花的秒數。


---

## 這台機器、Python 直譯器，以及 notebook 放在哪裡

這一格做三件事: 記下作業系統與 Python 的版本、找出教材根目錄 (放 `data/` 與 `environments/` 的那一層)、確認這一本跟兩本 lab 在同一個資料夾。

**資料夾放對是後面每一格的前提。** 兩本 lab 讀資料的寫法是「從現在的資料夾往上走，找到帶有 `data/synthetic` 的那一層」,所以這一本用同一套走法，並且把走出來的兩個路徑都印出來: 資料夾開錯的時候，路徑會直接顯示成別的地方。

In [ ]:
# 🎯 記下作業系統與 Python 版本，找出教材根目錄與 notebook 所在資料夾，並確認這一本放對位置。

# ---- 1. 這一格用到的標準函式庫 ----
#      這五個都是 Python 內建的，不需要另外安裝:
#      pathlib 處理檔案路徑、platform 讀作業系統資訊、sys 讀直譯器資訊、
#      os 讀環境變數、locale 讀作業系統偏好的文字編碼。
from pathlib import Path
import platform
import sys
import os
import locale
import logging      # 負責套件印出來的訊息，下面第 2 段會用到

# ---- 2. 先關掉 Prophet 這一串套件的訊息 ----
#      Prophet 與它的後端 cmdstanpy 在 import 與配適時會印出好幾行訊息 (例如找不到選配的
#      畫圖套件 plotly) 。那些訊息不影響結果，但會讓後面的表格夾在一堆文字中間，
#      所以在任何一格 import 它們之前先關掉 (兩本 lab 的第一格做的是同一件事)。
#      for 迴圈: 把括號裡三個名字輪流帶進 name，每一個都做同一件事。
for name in ("cmdstanpy", "prophet", "stan"):
    _logger = logging.getLogger(name)
    _logger.setLevel(logging.CRITICAL)      # 只有最嚴重的訊息才印
    _logger.handlers.clear()
    _logger.addHandler(logging.NullHandler())
    _logger.propagate = False

# ---- 3. pandas 與 display: 這一本所有的輸出都是表格 ----
#      pandas 是 Python 最常用的表格資料套件，這門課用它讀 CSV 與算統計量，
#      這一本只拿它把檢查結果排成一張表。
#      display() 來自 IPython，它會把表格畫成有框線的樣子，比 print 好讀。
#      這兩個 import 也包在 try 裡面: 萬一連 pandas 都沒裝，這一本仍然要跑得完，
#      只是輸出會退成純文字。
try:
    import pandas as pd
    from IPython.display import display
    HAS_PANDAS = True
    pd.set_option("display.max_columns", 60)
    pd.set_option("display.width", 200)
    pd.set_option("display.max_colwidth", 220)
except Exception:
    HAS_PANDAS = False


def show_table(rows, columns):
    """把幾列資料排成一張表印出來。rows 是「每一列是一個 tuple」的清單。"""
    if HAS_PANDAS:
        display(pd.DataFrame(rows, columns=columns))
    else:
        print(" | ".join(columns))
        for row in rows:
            print(" | ".join(str(x) for x in row))


# ---- 4. 存放檢查結果的地方 ----
# list 是 Python 的清單，用中括號建立，東西可以一個一個往後加。
# dict 是字典，用大括號建立，每一筆資料有自己的名字 (鍵) 與值。
# 底下每一格檢查完都會呼叫 record(),把一列結果加進 CHECKS 這個清單，
# 最後一格再把整個清單排成總表。
CHECKS = []


# def 是定義函式的語法。函式是一段取好名字、可以重複呼叫的程式。
def record(group, item, status, detail, fix=""):
    """把一項檢查的結果加進 CHECKS。status 只有三種: 通過 / 失敗 / 選配。"""
    CHECKS.append({"群組": group, "檢查項目": item, "狀態": status,
                   "實際結果": str(detail), "沒過的話怎麼辦": fix})
    return status, detail


def safe(group, item, func, fix=""):
    """執行一項檢查，並且保證它出錯的時候這一格不會中斷。

    try / except 是 Python 處理錯誤的語法: try 裡面的程式如果出錯，程式不會停下來，
    而是跳到 except 那一段。這一本每一項檢查都走這個函式，所以任何一項失敗
    都只會變成表格裡的一列。
    func 是傳進來的另一個函式，它要回傳 (狀態, 說明) 這兩個值。
    """
    try:
        status, detail = func()
    except Exception as exc:      # Exception 涵蓋所有一般的錯誤
        # f-string 是前面加上 f 的字串，大括號裡的東西會被換成它的值。
        # type(exc).__name__ 是錯誤的種類名稱，exc 是錯誤訊息本身。
        status, detail = "失敗", f"{type(exc).__name__}: {exc}"
    record(group, item, status, detail, fix)
    return status, detail


# ---- 5. 這台機器是什麼，以及這台機器該用哪一個環境檔 ----
OS_NAME = platform.system()          # Darwin 是 macOS，Windows 是 Windows，Linux 是 Linux
OS_LABEL = {"Darwin": "macOS", "Windows": "Windows", "Linux": "Linux"}.get(OS_NAME, OS_NAME)
ENV_YML = {"macOS": "environment.macos.yml",
           "Windows": "environment.windows.yml",
           "Linux": "environment.linux.yml"}.get(OS_LABEL, "environment.linux.yml")
ENV_NAME = "aiops-anomaly-zero-to-hero"        # 這門課的 conda 環境名稱
# 底下每一項「沒過的話怎麼辦」大多是這一行: 用這台機器對應的環境檔把套件補齊。
# 這裡刻意不加 --prune。--prune 的意思是「環境檔裡沒寫的套件就移掉」,而 week6 的環境檔
# 只列第六週用得到的套件,加了 --prune 會把前幾週其他 lab 需要的套件一起刪掉。
FIX_CONDA = (f"在教材資料夾裡打這兩行: conda activate {ENV_NAME} , "
             f"然後 conda env update -f environments/{ENV_YML}")

# ---- 6. 教材根目錄與 notebook 所在資料夾 ----
#      Path.cwd() 是「現在所在的資料夾」,也就是你開 JupyterLab 的那一層。
#      正常情況它就是 week6 資料夾本身: notebook 跟 data/ 放在同一層。
#      兩本 lab 也是從現在的資料夾一層一層往上走，找到帶有 data/synthetic 或
#      environments 的那一層，所以這一本走同一套，檢查到的路徑跟它們一定一致。
NB_DIR = Path.cwd()          # slides/、screenshots/、llm_diagnoses.json 都從這裡找
PROJECT_ROOT = NB_DIR
while (PROJECT_ROOT != PROJECT_ROOT.parent          # 還沒走到磁碟最上層
       and not (PROJECT_ROOT / "data" / "synthetic").is_dir()
       and not (PROJECT_ROOT / "environments").is_dir()):
    PROJECT_ROOT = PROJECT_ROOT.parent
if PROJECT_ROOT == PROJECT_ROOT.parent:             # 一路走到底都沒找到，退回原地
    PROJECT_ROOT = NB_DIR

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"      # 三份 CSV 放這裡
DATA_PROCESSED = PROJECT_ROOT / "outputs" / "workshop"    # 兩本 lab 的輸出寫這裡

# 兩本 lab 的檔名。這一本要跟它們在同一個資料夾。
LAB_NOTEBOOKS = ["2_lab06_forecasting.ipynb", "3_lab07_root_cause_analysis.ipynb"]


# ---- 7. 三項檢查 ----
def check_folder():
    # 這一行是 list comprehension，一行寫完的迴圈: 把 LAB_NOTEBOOKS 裡
    # 「在這個資料夾找不到」的檔名收成一個清單。
    missing = [name for name in LAB_NOTEBOOKS if not (NB_DIR / name).is_file()]
    if missing:
        return "失敗", (f"這個資料夾裡找不到 {', '.join(missing)}。"
                        f"現在的資料夾是 {NB_DIR}")
    if not DATA_SYNTHETIC.is_dir():
        return "失敗", f"兩本 lab 都在，但找不到資料夾 {DATA_SYNTHETIC}"
    return "通過", f"兩本 lab 都在 {NB_DIR},教材根目錄是 {PROJECT_ROOT}"


def check_python():
    version = platform.python_version()
    major_minor = sys.version_info[:2]          # 例如 (3, 12)
    if major_minor < (3, 10):
        return "失敗", f"Python {version} 太舊，教材的環境檔指定的是 Python 3.12"
    return "通過", f"Python {version}"


def check_kernel():
    # get_ipython 這個函式只有在 Jupyter kernel 裡才存在。
    # globals() 是「現在看得到的所有名字」,先問它在不在，直接呼叫的話沒有它會出錯。
    get_ipython = globals().get("get_ipython")
    if get_ipython is None:
        return "選配", "現在不是在 Jupyter kernel 裡執行，讀不到 kernel 資訊"
    shell = get_ipython()
    if shell is None:
        return "選配", "現在不是在 Jupyter kernel 裡執行，讀不到 kernel 資訊"
    # CONDA_DEFAULT_ENV 是 conda 設好的環境變數，值就是環境名稱。
    # 沒有這個變數的時候，改用直譯器所在資料夾的名字。
    env_name = os.environ.get("CONDA_DEFAULT_ENV") or Path(sys.prefix).name
    return "通過", f"kernel 類別 {type(shell).__name__},執行環境名稱 {env_name}"


GROUP = "機器與 Python"
safe(GROUP, "notebook 放在對的資料夾", check_folder,
     "把 1_pkg_checker.ipynb 與兩本 lab、data/、outputs/ 放回同一個 week6 資料夾，"
     "然後在 week6 這一層重開 JupyterLab")
safe(GROUP, "Python 版本", check_python, FIX_CONDA)
safe(GROUP, "Jupyter kernel", check_kernel,
     f"用 conda activate {ENV_NAME} 進到課程環境之後再開 JupyterLab")

# ---- 8. 輸出: 這台機器實際的樣子 ----
show_table([
    ("作業系統", platform.platform()),
    ("CPU 架構", platform.machine()),
    ("Python 版本", platform.python_version()),
    ("Python 直譯器位置", sys.executable),
    ("這台機器該用的環境檔", f"environments/{ENV_YML}"),
    ("現在所在的資料夾", str(NB_DIR)),
    ("教材根目錄", str(PROJECT_ROOT)),
    ("資料夾位置判定", CHECKS[0]["狀態"] + ": " + CHECKS[0]["實際結果"]),
], ["項目", "這台機器"])

---

## Docker (最久的一項,所以放在前面)

**Lab 07 最後那一段要把兩本 lab 的結果放到 Grafana 上看,那一段需要 Docker。** 它用 `python 4_grafana.py` 一行指令啟動四個容器 (兩支重播程式、Prometheus、Grafana) ,學員不用自己打任何 docker 指令。

**這一組標成「選配」的意思很窄: 缺了不會擋住前面任何一格分析。** 真的來不及裝,那一段改看講師的畫面就好,`screenshots/` 裡也有接好之後的樣子。但它是課程內容,不是加分題,所以能裝就裝。

**這一組放在第二個檢查,是因為它修起來最久。** 下面這一格如果告訴你 Docker 不能用,請**立刻**照它印出來的步驟開始下載安裝,然後回來繼續看後面的檢查結果,不要等整本跑完再開始。

這一格檢查三件事,三件是不同的問題,修法也不一樣:

1. **`docker` 這個指令在不在** (完全沒裝)
2. **Docker 的背景服務有沒有在跑** (裝好了但沒打開 Docker Desktop,這是最常見的)
3. **那四個 host port 有沒有被別的程式佔住** (通常是前幾週自己裝的 Grafana 或 Prometheus 還開著)


In [ ]:
# 🎯 檢查 docker 指令、Docker 背景服務，以及四個 host port 是否空著。

import shutil       # 內建套件，which() 用來找一個指令在不在系統的搜尋路徑上
import socket       # 內建套件，這裡拿它來試綁 port
import subprocess   # 內建套件，用來執行外部指令並拿回輸出

# 底下的修復步驟三個作業系統不一樣，先判斷這一台是哪一種。
IS_WINDOWS = platform.system() == "Windows"
IS_MACOS = platform.system() == "Darwin"

# ⚙️ 可調參數: WEEK6_PORTS
#    這四個 port 跟 4_grafana.py 的 DEFAULT_PORTS 是同一組，改了要兩邊一起改，
#    不然 notebook 檢查的 port 跟實際啟動的 port 會不一樣。兩支重播程式各佔一個:
#    8011 重播 Lab 06 的那兩晚，8010 重播 Lab 07 的事故 L。
#    被佔住的時候不用改這裡，改 infra/stack/.env 裡的 WEEK6_<名稱>_PORT 就好。
WEEK6_PORTS = {"Lab 06 重播程式": 8011, "Lab 07 重播程式": 8010,
               "Prometheus": 9090, "Grafana": 3000}

# ⚙️ 可調參數: DOCKER_TIMEOUT = 20
#    等 docker info 回應的秒數。Docker Desktop 正在啟動的期間這個指令要等很久才回應，
#    設太短會把「還在啟動」誤判成「沒有在跑」,設太長則會讓這一格停在這裡等。
DOCKER_TIMEOUT = 20

GROUP = "Docker 與 port"
rows = []


def check_docker_cli():
    if shutil.which("docker") is None:
        return "選配", "這台機器上沒有 docker 指令"
    return "選配", f"docker 指令在 {shutil.which('docker')}"


def check_docker_daemon():
    if shutil.which("docker") is None:
        return "選配", "沒有 docker 指令，跳過這一項"
    # capture_output=True 表示把指令的輸出接回來，text=True 表示當成文字而不是位元組。
    probe = subprocess.run(["docker", "info"], capture_output=True, text=True,
                           timeout=DOCKER_TIMEOUT)
    if probe.returncode != 0:
        return "選配", "docker 裝好了，但背景服務沒有在跑，要先打開 Docker Desktop"
    return "選配", "docker 背景服務有回應，最後的 Grafana 那一段可以跑"


def port_is_free(port):
    """試著綁一次這個 port。綁得起來代表沒有別的程式佔著它。"""
    # with 會在區塊結束時自動把 socket 關掉，不用自己記得收尾。
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        try:
            probe.bind(("0.0.0.0", port))      # Docker 也是綁這個位址，所以照它的方式試
        except OSError:
            return False
    return True


status, detail = safe(GROUP, "docker 指令", check_docker_cli,
                      "要跑 Lab 07 的 Grafana 那一段才需要: macOS 與 Windows 裝 Docker Desktop，"
                      "Linux 裝 Docker Engine")
rows.append(("docker 指令", status, detail))

status, detail = safe(GROUP, "Docker 背景服務", check_docker_daemon,
                      "打開 Docker Desktop，等它的圖示變成執行中再重跑這一格")
rows.append(("Docker 背景服務", status, detail))

for label, port in WEEK6_PORTS.items():
    status, detail = safe(
        GROUP, f"{label} 的 port {port}",
        lambda p=port: ("選配", f"port {p} 沒有被佔用") if port_is_free(p)
        else ("選配", f"port {p} 被別的程式佔住了"),
        f"關掉佔用的程式，或在 infra/stack/.env 設一個沒被佔用的 port。"
        f"前幾週用 brew 或 systemd 裝的 Prometheus 與 Grafana 會佔住 9090 與 3000")
    rows.append((f"{label} port {port}", status, detail))

show_table(rows, ["項目", "狀態", "實際結果"])

# ---- 這一組沒過的話，把要做的事直接印出來，不要等到最後一格 ----
#      理由是這一組修起來最久 (要下載安裝) ，越早開始越好。
#      any() 的意思是「清單裡只要有一個成立就算成立」。
_cli_ok = shutil.which("docker") is not None
_daemon_ok = _cli_ok and any(r[0] == "Docker 背景服務" and "有回應" in str(r[2]) for r in rows)
_busy = [r[0] for r in rows if "port" in r[0] and "佔住" in str(r[2])]

if not _cli_ok:
    print("\n" + "=" * 68)
    print("Docker 還沒裝。這是這一本裡修起來最久的一項，請現在就開始:")
    print("=" * 68)
    if IS_WINDOWS:
        print("  1. 開瀏覽器到 https://www.docker.com/products/docker-desktop/")
        print("  2. 下載 Docker Desktop for Windows，執行安裝檔，照精靈按到底")
        print("  3. 安裝完會要求重新開機，重開之後從開始功能表打開 Docker Desktop")
        print("  4. 等右下角的鯨魚圖示不再跑動，再回到這一格按 Shift+Enter 重跑")
    elif IS_MACOS:
        print("  1. 開瀏覽器到 https://www.docker.com/products/docker-desktop/")
        print("     晶片是 Apple Silicon (M 系列) 就選 Apple Chip，Intel 就選 Intel Chip")
        print("     這一台是:", platform.machine())
        print("  2. 打開下載的 .dmg，把 Docker 拖進 Applications")
        print("  3. 從 Applications 打開 Docker，第一次會要求輸入密碼授權")
        print("  4. 等選單列的鯨魚圖示不再跑動，再回到這一格按 Shift+Enter 重跑")
    else:
        print("  依發行版安裝 Docker Engine，例如 Ubuntu: https://docs.docker.com/engine/install/ubuntu/")
        print("  裝完把自己加進 docker 群組: sudo usermod -aG docker $USER，然後重新登入")
    print("  下載大約 600 MB 到 1 GB，視網路 10 到 30 分鐘。")
    print("  一邊下載一邊往下看後面的檢查，不用停在這裡等。")
    print("=" * 68)
elif not _daemon_ok:
    print("\n" + "=" * 68)
    print("Docker 裝好了，但背景服務沒有在跑。這一項通常一分鐘內就能修好:")
    print("=" * 68)
    print("  1. 打開 Docker Desktop (Windows 從開始功能表，macOS 從 Applications) ")
    print("  2. 等圖示不再跑動，顯示執行中")
    print("  3. 回到這一格按 Shift+Enter 重跑")
    print("  打開之後還是不行，先結束 Docker Desktop 再開一次;")
    print("  兩次都不行就重新安裝一次，那是這一項唯一剩下的做法。")
    print("=" * 68)

if _busy:
    print("\n" + "-" * 68)
    print("下面這幾個 port 被別的程式佔住了:", "、".join(_busy))
    print("兩個做法二選一:")
    print("  (a) 關掉佔用的程式。最常見的是前幾週用 brew 或 systemd 裝的 Grafana 與 Prometheus:")
    print("      macOS: brew services stop grafana   與   brew services stop prometheus")
    print("      Linux: sudo systemctl stop grafana-server   與   sudo systemctl stop prometheus")
    print("  (b) 換一個 port。到 infra/stack/ 把 .env.example 複製成 .env，")
    print("      改裡面的 WEEK6_GRAFANA_PORT 之類的那幾行，例如改成 3001。")
    print("-" * 68)

---

## 兩本 lab 需要的套件

這一格把每一個套件真的 import 一次，並印出版本。**安裝時打的名字跟 import 時打的名字不一定一樣**,表格因此分成兩欄:

| 安裝時打 | import 時打 |
| --- | --- |
| `scikit-learn` | `sklearn` |
| `ipython` | `IPython` |

`conda install sklearn` 會找不到套件，`import scikit-learn` 則是語法錯誤，兩個名字要各用在各自的地方。

In [ ]:
# 🎯 逐一 import 兩本 lab 用到的套件並印出版本，缺哪一個就在表格裡標成失敗。

# importlib 是 Python 內建的套件，import_module() 讓我們用「字串」指定要 import 什麼。
# 這一本刻意全部走這個寫法: 用一般的 import 寫在最上面的話，缺套件會讓整本停在第一格，
# 缺什麼就看不到了; 走 import_module 才能把「缺這一個」變成表格裡的一列。
import importlib
from importlib import metadata      # metadata 可以查已安裝套件的版本

# 每一列是 (安裝時打的名字, import 時打的名字, 這門課拿它做什麼)。
REQUIRED = [
    ("numpy", "numpy",
     "數值運算的基礎套件，所有陣列與數學運算都靠它"),
    ("pandas", "pandas",
     "表格資料套件，讀 CSV、篩選、算統計量都用它"),
    ("scipy", "scipy",
     "科學計算套件，兩本 lab 用它的統計函式，例如常態分布的分位數"),
    ("matplotlib", "matplotlib",
     "畫圖套件，兩本 lab 的圖都是它畫的"),
    ("scikit-learn", "sklearn",
     "Python 最常用的機器學習套件，這門課只用到它的其中一個模型 HistGradientBoostingRegressor"),
    ("statsmodels", "statsmodels",
     "統計模型套件，Lab 06 的 Holt-Winters 與 SARIMA 基準線用它"),
    ("prophet", "prophet",
     "時間序列預測套件，Lab 06 的主模型，負責學「這個時刻的正常是多少」"),
    ("networkx", "networkx",
     "處理圖 (節點與邊) 的套件，Lab 07 用它算拓樸上的下游可達範圍"),
    ("prometheus_client", "prometheus_client",
     "把數值輸出成 Prometheus 讀得懂的格式，Lab 07 的 Grafana 重播服務用它"),
    ("ipython", "IPython",
     "Jupyter 的互動介面，display() 這個函式來自它"),
    ("ipykernel", "ipykernel",
     "讓 JupyterLab 可以在這個 Python 環境裡執行程式的套件"),
]


def package_version(module, install_name):
    """先看套件自己的 __version__,沒有的話再去問已安裝套件的中繼資料。"""
    version = getattr(module, "__version__", None)      # 大多數套件都有這個屬性
    if version:
        return str(version)
    try:
        return metadata.version(install_name)           # 例如 prometheus_client 就走這一條
    except Exception:
        return "讀不到版本"


GROUP = "套件"
rows = []
for install_name, import_name, purpose in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = package_version(module, install_name)
        status, detail = "通過", f"{version}"
    except Exception as exc:
        status, detail = "失敗", f"import {import_name} 失敗: {type(exc).__name__}: {exc}"
    record(GROUP, f"{install_name} (import {import_name})", status, detail, FIX_CONDA)
    rows.append((install_name, import_name, status, detail, purpose))

show_table(rows, ["安裝時打的名字", "import 時打的名字", "狀態", "版本或錯誤訊息", "這門課拿它做什麼"])

---

## Prophet 與 scikit-learn 實際配適一次

上一格只確認 import 得起來，這一格再往前一步: 拿一小段假資料實際配適一次。這兩個套件裝得起來、import 得起來，仍然可能在配適那一步才出錯:

- **Prophet** 的計算是交給 cmdstanpy 這個後端做的，它需要一個編譯好的模型檔。這一格會印出這台機器上實際花了幾秒。
- **scikit-learn** 這一格配適的是 `HistGradientBoostingRegressor`,那正是 Lab 06 拿來學殘差的模型。

兩項都通過，代表 Lab 06 的兩個模型在這台機器上跑得動。

In [ ]:
# 🎯 用一小段假資料實際配適 Prophet 與 HistGradientBoostingRegressor，並量出各花幾秒。

import time      # 內建套件，time.time() 回傳現在的時刻，兩次相減就是經過的秒數

# ⚙️ 可調參數: TOY_ROWS = 40
#    假資料的列數。調高會讓配適變慢，調低到 2 以下 Prophet 會拒絕配適。
#    選 40 是因為它相當於五週多的日資料，足夠讓 Prophet 估得出週季節性，
#    又遠小於上課用的 43,200 列，所以這一格量到的是「這台機器跑不跑得動」,
#    不是「上課要等多久」。
TOY_ROWS = 40

# ⚙️ 可調參數: TOY_MAX_ITER = 30
#    HistGradientBoostingRegressor 要疊幾棵樹 (預設 100) 。這一格只要確認它配適得起來，
#    30 棵就夠; 調高會變慢，調低到 1 仍然跑得動但等於沒有在學東西。
TOY_MAX_ITER = 30


def fit_prophet():
    # import_module 用字串指定套件，回傳的東西後面接屬性名稱就拿到裡面的類別。
    Prophet = importlib.import_module("prophet").Prophet
    np = importlib.import_module("numpy")

    # Prophet 規定輸入的表格只有兩欄，而且名字固定: ds 是時間，y 是要預測的值。
    days = pd.date_range("2026-01-01", periods=TOY_ROWS, freq="D")
    trend = np.linspace(10.0, 20.0, TOY_ROWS)                       # 從 10 線性長到 20
    weekly = np.sin(np.arange(TOY_ROWS) / 7.0 * 2 * np.pi)          # 一週一輪的起伏
    toy = pd.DataFrame({"ds": days, "y": trend + weekly})

    started = time.time()
    model = Prophet(weekly_seasonality=True, yearly_seasonality=False, daily_seasonality=False)
    model.fit(toy)
    future = model.make_future_dataframe(periods=3, freq="D")       # 往後多要三天
    forecast = model.predict(future)
    seconds = time.time() - started

    # .iloc[-1] 是「最後一列」,yhat 是 Prophet 給的預測值。
    last = float(forecast["yhat"].iloc[-1])
    return "通過", (f"配適 {TOY_ROWS} 列並預測 3 天成功，花 {seconds:.1f} 秒，"
                    f"最後一天的預測值 {last:.2f}")


def fit_sklearn():
    HGBR = importlib.import_module("sklearn.ensemble").HistGradientBoostingRegressor
    np = importlib.import_module("numpy")

    # default_rng(0) 是固定亂數種子的亂數產生器，每次跑出來的假資料都一樣。
    rng = np.random.default_rng(0)
    n_rows = TOY_ROWS * 5
    X = rng.normal(size=(n_rows, 3))                         # 三欄特徵
    y = X[:, 0] * 2.0 + X[:, 1] + rng.normal(scale=0.1, size=n_rows)   # 要學的答案

    started = time.time()
    model = HGBR(max_iter=TOY_MAX_ITER, random_state=0).fit(X, y)
    predicted = model.predict(X[:5])                         # 拿前五列回頭預測一次
    seconds = time.time() - started
    return "通過", (f"配適 {n_rows} 列 3 欄成功，花 {seconds:.2f} 秒，"
                    f"前五個預測值 {np.round(predicted, 2).tolist()}")


GROUP = "模型配適"
safe(GROUP, "Prophet 配適與預測", fit_prophet,
     f"{FIX_CONDA}; 更新之後仍然失敗的話，把整段錯誤訊息複製給講師")
safe(GROUP, "scikit-learn HistGradientBoostingRegressor 配適", fit_sklearn, FIX_CONDA)

# CHECKS[-2] 與 CHECKS[-1] 是剛剛加進去的那兩列。
show_table([(c["檢查項目"], c["狀態"], c["實際結果"]) for c in CHECKS[-2:]],
           ["項目", "狀態", "實際結果"])

---

## matplotlib 的中文字型

兩本 lab 的圖，標題與座標軸標籤都是中文。matplotlib 內定的字型沒有中文字，沒設定的話每一個中文字都會畫成空心方框。

**這一格會畫一張小圖出來，請用眼睛確認圖上是中文字而不是方框。** 版本號與檔案存在與否可以用程式判斷，字型有沒有真的被畫出來不行，所以這一項要你自己看。

In [ ]:
# 🎯 找出這台機器上可用的中文字型，設給 matplotlib，然後畫一張中文圖出來讓你用眼睛確認。

# 三個作業系統內建的中文字型名稱不一樣，所以照順序找第一個裝得到的
# (這份候選清單跟兩本 lab 第一格用的完全相同) 。
CJK_CANDIDATES = ["PingFang TC", "PingFang HK", "Heiti TC", "Arial Unicode MS",   # macOS
                  "Microsoft JhengHei", "Microsoft YaHei",                        # Windows
                  "Noto Sans CJK TC", "Noto Sans TC", "Source Han Sans TW",       # Linux
                  "WenQuanYi Zen Hei", "Droid Sans Fallback"]

FONT_FIX = {
    "macOS": "系統內建 PingFang TC。找不到的話先刪掉字型快取資料夾 ~/.matplotlib 再重跑一次",
    "Windows": "系統內建微軟正黑體 Microsoft JhengHei。找不到的話先刪掉字型快取資料夾 "
               "%USERPROFILE%\\.matplotlib 再重跑一次",
    "Linux": "先安裝思源黑體: sudo apt install fonts-noto-cjk , 然後刪掉 ~/.matplotlib 再重跑一次",
}.get(OS_LABEL, "安裝一套中文字型，然後刪掉 matplotlib 的字型快取資料夾再重跑一次")

CJK_FONT = None


def check_font():
    """找字型、設給 matplotlib、畫一張中文圖。找不到字型也照畫，讓你看到方框長什麼樣子。"""
    global CJK_FONT           # global 的意思是這裡改的是外面那個 CJK_FONT，不是新開一個
    plt = importlib.import_module("matplotlib.pyplot")
    fm = importlib.import_module("matplotlib.font_manager")

    # 大括號加 for 是 set comprehension，把系統裝的每一個字型檔的名字收成一個集合。
    installed = {f.name for f in fm.fontManager.ttflist}
    # next(...) 取出第一個符合條件的候選字型; 一個都沒有就回傳 None。
    CJK_FONT = next((name for name in CJK_CANDIDATES if name in installed), None)
    if CJK_FONT:
        plt.rcParams["font.sans-serif"] = [CJK_FONT] + plt.rcParams["font.sans-serif"]
        # 中文字型多半沒有獨立的粗體檔，matplotlib 會為每一次粗體字送出一行提示訊息。
        # 它不影響輸出，但會讓這一格多出很多行文字，所以關掉。
        logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
    plt.rcParams["axes.unicode_minus"] = False     # 用 ASCII 的減號，中文字型才畫得出負號

    fig, ax = plt.subplots(figsize=(6.6, 2.6))
    ax.plot([0, 1, 2, 3, 4], [1.0, 3.0, 2.0, 4.0, 3.5],
            color="tab:red", marker="o", label="流量")
    ax.set_title("中文字型測試: 這一行要看得到字，不能是空心方框")
    ax.set_xlabel("時間 (小時)")
    ax.set_ylabel("流量 (bytes)")
    ax.legend(loc="upper left")
    # transform=ax.transAxes 的意思是這一行字的位置用「圖框的比例」定位，
    # 0.60 與 0.08 就是從左邊算 60%、從下面算 8% 的地方，不會壓到上面那條線。
    ax.text(0.60, 0.08, "負號要畫得出來: -12.5", transform=ax.transAxes)
    fig.tight_layout()
    display(fig)          # 把圖印在這一格底下
    plt.close(fig)        # 關掉圖，不然 matplotlib 會在下一格再印一次

    if CJK_FONT is None:
        return "失敗", "候選清單裡的中文字型一個都沒裝到，上面那張圖的中文會是空心方框"
    return "通過", f"用的是 {CJK_FONT},請確認上面那張圖的中文有顯示出來"


GROUP = "中文字型"
safe(GROUP, "matplotlib 畫得出中文", check_font, FONT_FIX)

show_table([(CHECKS[-1]["檢查項目"], CHECKS[-1]["狀態"], CHECKS[-1]["實際結果"])],
           ["項目", "狀態", "實際結果"])

---

## 三份資料檔

兩本 lab 讀的是同一份 Week 6 資料集: 指標檔、事件目錄、排程行事曆。這一格對每一份印出檔案大小、列數、欄數，指標檔另外印出時間範圍與 port 的個數，最後把每一份的前三列顯示出來。

**前三列要看。** 檔案存在不代表內容是對的: 用 Excel 開過再存檔會改掉欄位格式與編碼，下載或解壓縮中斷會留下長度不足的檔案。這一格因此報的是列數、欄數與前三列的實際內容，不只是檔案在不在。

In [ ]:
# 🎯 檢查三份 CSV 的大小、列數、欄數與時間範圍，並顯示每一份的前三列。

# 每一列是 (檔名, 這一份是什麼)。三份都放在 data/synthetic/ 底下。
DATA_FILES = [
    ("synthetic_rrd_metrics_week6.csv",
     "指標檔: 五個 port、一整個月、每 5 分鐘一列的原始計數器"),
    ("synthetic_event_catalog_week6.csv",
     "事件目錄: 每一次注入的事件是什麼型別、在哪個 port、從何時到何時"),
    ("synthetic_scheduled_calendar_week6.csv",
     "排程行事曆: 開盤微爆、收盤集合競價、日終對帳這些事先就知道會發生的行為"),
]

GROUP = "資料檔"
rows = []
loaded = {}      # 讀成功的表格先收在這裡，等一下顯示前三列要用

for filename, purpose in DATA_FILES:
    path = DATA_SYNTHETIC / filename

    def read_one(path=path, filename=filename):
        # 預設參數 path=path 的用意是把現在這一圈的 path 固定住，
        # 不然函式要到後面才執行，會拿到迴圈最後一圈的值。
        if not path.is_file():
            return "失敗", f"找不到檔案 {path}"
        size_bytes = path.stat().st_size                   # st_size 的單位是 byte
        if size_bytes == 0:
            return "失敗", f"{path} 的大小是 0,檔案是空的"
        # 大於 1 MB 的檔案報 MB，小的報 KB，不然小檔會顯示成 0.00 MB 看不出差別。
        size_text = (f"{size_bytes / 1024 / 1024:.2f} MB" if size_bytes >= 1024 * 1024
                     else f"{size_bytes / 1024:.1f} KB")
        table = pd.read_csv(path)
        loaded[filename] = table
        detail = f"{size_text},{len(table):,} 列，{table.shape[1]} 欄"
        # 指標檔多報兩件事: 時間範圍與 port 個數。
        if "timestamp" in table.columns:
            stamps = pd.to_datetime(table["timestamp"])
            detail += (f",時間 {stamps.min():%Y-%m-%d %H:%M} 到 {stamps.max():%Y-%m-%d %H:%M}"
                       f",{table['port_id'].nunique()} 個 port")
        return "通過", detail

    status, detail = safe(GROUP, filename, read_one,
                          "重新下載一次教材資料夾，把 data/synthetic/ 整個換掉")
    rows.append((filename, status, detail, purpose))

show_table(rows, ["檔名", "狀態", "實際結果", "這一份是什麼"])

# 前三列: 看得到欄名與實際的數字，才知道檔案內容沒有被改掉。
if HAS_PANDAS:
    for filename, table in loaded.items():
        print(f"{filename} 的前 3 列")
        display(table.head(3))

---

## 寫入權限

兩本 lab 會把結果寫成檔案: Lab 07 寫出的 `outputs/workshop/rca_case_L.csv` 就是最後 Grafana 重播讀的那一份。這一格在那個資料夾建立一個小檔案、讀回來、再刪掉，確認這台機器寫得進去。

唯讀的位置寫不進去: 還掛載著的 `.dmg` 磁碟映像檔、直接從壓縮檔裡點開的暫存資料夾，都屬於這一種。

In [ ]:
# 🎯 在 outputs/workshop/ 建立一個小檔案、讀回來、再刪掉，確認這台機器寫得進去。

def check_write():
    # mkdir 建立資料夾。parents=True 的意思是路徑中間缺的層一起建，
    # exist_ok=True 的意思是已經存在就當作成功，不要當成錯誤。
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    probe = DATA_PROCESSED / "_1_pkg_checker_write_test.txt"
    content = "這是環境檢查寫出來的測試檔，寫完就會被刪掉。"
    probe.write_text(content, encoding="utf-8")       # 明講 utf-8，中文才寫得進去
    read_back = probe.read_text(encoding="utf-8")
    probe.unlink()                                     # unlink 就是刪檔案
    if read_back != content:
        return "失敗", f"寫進去的字跟讀回來的不一樣，讀回來的是 {read_back!r}"
    return "通過", f"{DATA_PROCESSED} 可以寫入，測試檔已刪除"


GROUP = "寫入權限"
safe(GROUP, "outputs/workshop 可以寫入", check_write,
     "把整個教材資料夾複製到桌面或家目錄底下再解壓縮一次，不要留在下載資料夾或掛載的磁碟映像檔裡")

show_table([(CHECKS[-1]["檢查項目"], CHECKS[-1]["狀態"], CHECKS[-1]["實際結果"])],
           ["項目", "狀態", "實際結果"])

---

## 教材檔案清點

兩本 lab 除了資料以外還會讀這些東西: `slides/` 裡理論課的投影片圖、`screenshots/` 裡的 Grafana 畫面、`llm_diagnoses.json` (Lab 07 用它重播一次真實的 LLM 診斷，所以不需要 API key)、以及兩支 Python 程式。

少了圖只會讓 notebook 該有圖的地方顯示成一個讀不到圖片的圖示，不會中斷; 少了 `llm_diagnoses.json` 或 `4_grafana.py` 則會讓 Lab 07 的後半段跑不完。

In [ ]:
# 🎯 清點投影片圖、螢幕截圖、JSON 與兩支 Python 程式，缺什麼就把檔名列出來。

# ⚙️ 可調參數: EXPECTED_SLIDES = 23 , EXPECTED_SHOTS = 2
#    這兩個數字是這一版教材實際的圖檔數量。教材換版、投影片增減的時候要跟著改，
#    改小了會漏掉缺檔，改大了會讓每個人都看到失敗。
EXPECTED_SLIDES = 23
EXPECTED_SHOTS = 2

# 一定要在 notebook 同一個資料夾裡的檔案。
LOCAL_FILES = [
    ("llm_diagnoses.json", "Lab 07 錄好的 LLM 診斷內容，沒有它最後兩節跑不完"),
    ("4_grafana.py", "Lab 07 的 Grafana 環境啟動程式"),
    ("results_exporter.py", "把 notebook 的結果輸出成 Grafana 讀得到的格式"),
]
# 螢幕截圖的檔名是寫死在 notebook 的 markdown 裡的，所以逐個確認。
SHOT_FILES = ["lab06_grafana_forecast_row.png", "lab07_grafana_rca_row.png"]

GROUP = "教材檔案"
rows = []


def count_png(folder, expected):
    """數一個資料夾裡有幾個 .png。glob 是「照樣式找檔案」,*.png 代表所有 png 檔。"""
    if not folder.is_dir():
        return "失敗", f"找不到資料夾 {folder}"
    found = len(list(folder.glob("*.png")))
    if found < expected:
        return "失敗", f"只找到 {found} 張 png,這一版教材應該有 {expected} 張"
    return "通過", f"找到 {found} 張 png (這一版教材至少要有 {expected} 張)"


status, detail = safe(GROUP, "slides/ 的投影片圖",
                      lambda: count_png(NB_DIR / "slides", EXPECTED_SLIDES),
                      "重新解壓縮一次教材資料夾，把 slides/ 整個換掉")
rows.append(("slides/", status, detail))

status, detail = safe(GROUP, "screenshots/ 的 Grafana 畫面",
                      lambda: count_png(NB_DIR / "screenshots", EXPECTED_SHOTS),
                      "重新解壓縮一次教材資料夾，把 screenshots/ 整個換掉")
rows.append(("screenshots/", status, detail))

# lambda 是「只有一行的匿名函式」,這裡用它把每一個檔案的檢查包成一個函式傳給 safe。
for filename, purpose in LOCAL_FILES:
    path = NB_DIR / filename
    status, detail = safe(
        GROUP, filename,
        lambda p=path: ("通過", f"{p.stat().st_size:,} bytes") if p.is_file()
        else ("失敗", f"找不到 {p}"),
        "重新解壓縮一次教材資料夾")
    rows.append((filename, status, detail))

for filename in SHOT_FILES:
    path = NB_DIR / "screenshots" / filename
    status, detail = safe(
        GROUP, f"screenshots/{filename}",
        lambda p=path: ("通過", "在") if p.is_file() else ("失敗", f"找不到 {p}"),
        "重新解壓縮一次教材資料夾，把 screenshots/ 整個換掉")
    rows.append((f"screenshots/{filename}", status, detail))

# 這一份不在 notebook 旁邊，它在教材根目錄底下的 infra/stack/。
compose_path = PROJECT_ROOT / "infra" / "stack" / "compose.yaml"
status, detail = safe(
    GROUP, "infra/stack/compose.yaml",
    lambda p=compose_path: ("通過", f"{p}") if p.is_file() else ("失敗", f"找不到 {p}"),
    "重新解壓縮一次教材資料夾，把 infra/ 整個換掉。這一份只有最後的 Grafana 那一段會用到")
rows.append(("infra/stack/compose.yaml", status, detail))

show_table(rows, ["項目", "狀態", "實際結果"])

---

## JupyterLab 版本與 mermaid 圖

兩本 lab 的說明文字裡有流程圖，寫法是 markdown 的 ```` ```mermaid ```` 區塊。**JupyterLab 從 4.1 開始才會把它畫成圖**,更舊的版本會原樣顯示成一堆文字。

同一件事也發生在別的開啟方式上:

- **VS Code** 的 notebook 介面不會把 notebook markdown 裡的 mermaid 畫成圖
- 舊的 **nbclassic** 介面 (`jupyter notebook` 開起來、網址是 `/tree` 的那個) 也不會

這兩種情況下 lab 仍然跑得動，只是流程圖會變成文字。這門課請用 JupyterLab 開。

In [ ]:
# 🎯 讀出這個環境裡的 JupyterLab 版本，判斷它畫不畫得出 markdown 裡的 mermaid 流程圖。

# ⚙️ 可調參數: MERMAID_MIN = (4, 1)
#    JupyterLab 內建 mermaid 支援的最低版本。這是 JupyterLab 4.1 的變更，
#    未來如果改用別的畫圖套件才需要動這個值。
MERMAID_MIN = (4, 1)


def check_jupyterlab():
    try:
        version = metadata.version("jupyterlab")
    except Exception:
        # 學生有可能是從另一個環境啟動 JupyterLab，再選這個環境的 kernel 來跑。
        # 那種情況下這個環境裡查不到 jupyterlab，不代表版本不夠。
        return "選配", ("這個環境裡沒有 jupyterlab 套件，量不到版本。"
                        "請改看畫面左上角 Help > About JupyterLab 顯示的版本")
    # 版本字串像 4.6.3,split(".") 用句點切開，取前兩段轉成數字來比大小。
    parts = version.split(".")
    numeric = tuple(int(p) for p in parts[:2] if p.isdigit())
    if len(numeric) < 2:
        return "選配", f"版本字串是 {version},判讀不出主版本號"
    if numeric < MERMAID_MIN:
        return "失敗", (f"JupyterLab {version} 比 {MERMAID_MIN[0]}.{MERMAID_MIN[1]} 舊，"
                        f"兩本 lab 的 mermaid 流程圖會顯示成文字")
    return "通過", f"JupyterLab {version},畫得出 mermaid 流程圖"


GROUP = "JupyterLab"
safe(GROUP, "版本足以顯示 mermaid 流程圖", check_jupyterlab,
     f"conda activate {ENV_NAME} 之後跑 conda install -c conda-forge \"jupyterlab>=4.1\"; "
     f"另外請用 JupyterLab 開這兩本 lab，VS Code 與舊的 nbclassic 介面不會把 mermaid 畫成圖")

show_table([(CHECKS[-1]["檢查項目"], CHECKS[-1]["狀態"], CHECKS[-1]["實際結果"])],
           ["項目", "狀態", "實際結果"])

---

## 文字編碼

教材的三份 CSV 與 `llm_diagnoses.json` 都是 UTF-8,裡面有中文。Python 讀檔時如果沒有指定編碼，會用作業系統偏好的編碼去讀，所以這一格把兩個編碼設定印出來。

macOS 與 Linux 的偏好編碼是 UTF-8。**Windows 沒有開啟 UTF-8 支援時，偏好編碼會是系統的 ANSI 字碼頁** (繁體中文版是 cp950),用那個編碼讀教材的中文檔會出現 `UnicodeDecodeError`,所以這一格在 Windows 上量到非 UTF-8 就標成失敗。

In [ ]:
# 🎯 印出 Python 內定編碼與作業系統偏好編碼，Windows 上不是 UTF-8 就標成失敗。

def check_encoding():
    default_enc = sys.getdefaultencoding()               # Python 內部字串轉位元組的內定編碼
    # locale 是「地區設定」,getpreferredencoding 回傳作業系統偏好的檔案編碼。
    # 參數 False 的意思是不要重新去問作業系統一次，直接用啟動時讀到的值。
    preferred = locale.getpreferredencoding(False)
    # .lower() 轉小寫，.replace() 把減號去掉，這樣 UTF-8 與 utf8 會被當成同一個。
    normalised = preferred.lower().replace("-", "")
    if normalised != "utf8":
        if OS_LABEL == "Windows":
            return "失敗", (f"偏好編碼是 {preferred},不是 UTF-8。"
                            f"讀教材的中文 CSV 與 JSON 會出現 UnicodeDecodeError")
        return "失敗", f"偏好編碼是 {preferred},不是 UTF-8"
    return "通過", f"內定 {default_enc},偏好 {preferred}"


GROUP = "文字編碼"
safe(GROUP, "檔案編碼是 UTF-8", check_encoding,
     "Windows: 在「設定 > 時間與語言 > 語言與地區 > 系統管理語言設定 > 變更系統地區設定」"
     "勾選 Beta 版 UTF-8,重開機; 或在開 JupyterLab 之前先設環境變數 PYTHONUTF8=1")

# stdout 的編碼跟上面兩個是不同的東西 (它管的是印到畫面上的字),
# 一併印出來，講師看回報內容時用得到。
show_table([
    ("sys.getdefaultencoding()", sys.getdefaultencoding()),
    ("locale.getpreferredencoding()", locale.getpreferredencoding(False)),
    ("畫面輸出編碼 sys.stdout.encoding", getattr(sys.stdout, "encoding", "讀不到")),
    ("環境變數 PYTHONUTF8", os.environ.get("PYTHONUTF8", "沒有設定")),
], ["項目", "這台機器"])

---

## 總表與回報用的純文字

上面每一格的結果都收在同一個清單裡，這一格把它們排成一張表，並且印出一段純文字。

**總表全部是通過或選配，就可以開始上課了。** 出現失敗的話，照那一列的「沒過的話怎麼辦」處理，處理完重跑一次 Run All Cells; 處理不掉就把下面那段純文字整段複製給講師，裡面已經有作業系統、Python 版本、資料夾位置，以及每一項失敗的原因。

In [ ]:
# 🎯 把每一項檢查排成總表，統計三種狀態各幾項，並印出一段可以複製給講師的純文字。

from datetime import datetime      # 內建套件，用來取現在的時間

# ---- 1. 統計三種狀態各有幾項 ----
# 這裡的中括號寫法一樣是 list comprehension，len() 是算清單裡有幾個東西。
passed = [c for c in CHECKS if c["狀態"] == "通過"]
failed = [c for c in CHECKS if c["狀態"] == "失敗"]
optional = [c for c in CHECKS if c["狀態"] == "選配"]

show_table([("通過", len(passed)), ("失敗", len(failed)), ("選配", len(optional))],
           ["狀態", "項數"])

# ---- 2. 要先動手的項目，照「修起來要多久」排 ----
#      失敗的項目排在最前面，而且不是照檢查順序排，是照修起來要多久排:
#      要下載東西的排前面，因為那些越早開始越好。
# ⚙️ 可調參數 FIX_COST: 每一組修起來的相對成本，數字小的排前面。
#    改這裡只會改「建議先做哪一項」的順序，不會改任何一項的判定結果。
#    新增一組檢查而沒有寫進這張表的話，它會拿到預設的 50，排在中間偏後。
FIX_COST = {
    "Docker 與 port": 0,      # 要下載安裝，10 到 30 分鐘
    "套件": 10,               # conda env update 要下載，5 到 20 分鐘
    "模型配適": 20,           # 多半是重裝 prophet
    "中文字型": 30,           # 裝一個字型檔
    "資料檔": 40,             # 重新解壓縮教材資料夾
    "教材檔案": 40,
    "JupyterLab": 45,
    "寫入權限": 45,
    "機器與 Python": 5,       # 開錯資料夾，關掉重開就好，所以其實最快
    "文字編碼": 60,
}
# sorted(..., key=...) 是照 key 算出來的值由小到大排。.get(x, 50) 是「查不到就用 50」。
todo = sorted(failed, key=lambda c: FIX_COST.get(c["群組"], 50))

if todo:
    print("=" * 68)
    print(f"有 {len(todo)} 項要處理。下面照「修起來要多久」排好了，請從第一項開始:")
    print("=" * 68)
    for i, check in enumerate(todo, start=1):
        print(f"{i}. [{check['群組']}] {check['檢查項目']}")
        print(f"   現在的狀況: {check['實際結果']}")
        print(f"   要做的事:   {check['沒過的話怎麼辦']}")
        print()
    print("需要下載的那幾項先開始跑，一邊等一邊修後面的。")
    print("=" * 68)
else:
    print("=" * 68)
    print("全部通過，兩本 lab 需要的東西這台機器上都有。可以開始上課。")
    if optional:
        print(f"另外有 {len(optional)} 項是選配 (Docker 那一組) 。"
              "它只影響最後的 Grafana 那一段，前面的分析不受影響。")
    print("=" * 68)

# ---- 3. 完整的總表 ----
#      上面只列要處理的，這一張是全部，含通過的項目，方便逐項對照。
if HAS_PANDAS:
    display(pd.DataFrame(CHECKS))
else:
    for check in CHECKS:
        print(check)

# ---- 4. 給講師的純文字 ----
#      這一段刻意用 print 而不是表格: 表格複製起來會變成沒有對齊的一堆字，純文字可以直接貼進訊息裡。
lines = [
    "===== 環境檢查結果，複製這一整段貼給講師 =====",
    f"檢查時間: {datetime.now():%Y-%m-%d %H:%M}",
    f"作業系統: {platform.platform()} ({platform.machine()})",
    f"Python: {platform.python_version()}",
    f"Python 直譯器: {sys.executable}",
    f"現在所在的資料夾: {NB_DIR}",
    f"教材根目錄: {PROJECT_ROOT}",
    f"中文字型: {CJK_FONT if CJK_FONT else '找不到'}",
    f"結果: 通過 {len(passed)} 項，失敗 {len(failed)} 項，選配 {len(optional)} 項",
    "",
]
if failed:
    lines.append("失敗的項目:")
    # enumerate 會一邊給編號一邊取出清單裡的東西，start=1 表示從 1 開始編。
    for i, check in enumerate(failed, start=1):
        lines.append(f"  {i}. [{check['群組']}] {check['檢查項目']}")
        lines.append(f"     實際結果: {check['實際結果']}")
        lines.append(f"     建議做法: {check['沒過的話怎麼辦']}")
else:
    lines.append("沒有失敗的項目，兩本 lab 需要的東西這台機器上都有。")
lines.append("=============================================")
print("\n".join(lines))       # "\n".join(...) 把清單裡的每一行用換行接起來